# Room Booking technical notebook

This notebook summarizes the technologies and implementation choices used for the Promtior technical challenge. The source code remains the executable source of truth; snippets below point to the same concepts used in the solution.

## Stack

- .NET 8 / ASP.NET Core minimal API
- PostgreSQL with EF Core and Npgsql
- Groq OpenAI-compatible Chat Completions API
- MSTest, FluentAssertions, and Testcontainers
- Docker Compose for local PostgreSQL

The application is a modular monolith: `Api → Application → Domain`, while Infrastructure implements ports for PostgreSQL, authentication, time configuration, and the LLM provider.

## Domain validation example

Bookings use UTC half-open intervals and 30-minute boundaries. The Domain is independent of ASP.NET Core, EF Core, and the LLM provider.

```csharp
// RoomBooking.Domain/Bookings/BookingPeriod.cs (conceptual excerpt)
var overlaps = startUtc < other.EndUtc && other.StartUtc < endUtc;
var alignsToSlot = value.TimeOfDay.Ticks % SlotDuration.Ticks == 0;

// Application creates the period before persistence.
var periodResult = BookingPeriod.Create(startUtc, endUtc);
```

## Concurrent booking protection

Application checks overlap to return a friendly error. PostgreSQL is the final authority when two requests race, using an exclusion constraint over active bookings.

```sql
ALTER TABLE bookings
ADD CONSTRAINT "EX_bookings_room_period_active"
EXCLUDE USING gist
(room_id WITH =, tstzrange(start_utc, end_utc, '[)') WITH &&)
WHERE (status = 1);
```

## Tool-calling boundary

The LLM interprets language and requests one of five typed tools. A tool delegates to an existing Application use case. It does not receive a user identifier: the server resolves identity through `ICurrentUser`.

```csharp
public interface IChatModel
{
    Task<ChatModelResponse> SendAsync(
        IReadOnlyList<ChatMessage> messages,
        IReadOnlyList<ChatToolDefinition> tools,
        CancellationToken cancellationToken = default);
}

// The provider adapter belongs to Infrastructure.
services.AddHttpClient<IChatModel, GroqChatModel>();
```

## Testing approach

Tests use MSTest and FluentAssertions. Domain and Application tests cover business invariants. API integration tests use a real PostgreSQL container. Agent tests use a deterministic `FakeChatModel`, so CI never depends on a Groq API key, Internet access, provider availability, or non-deterministic model output.